In [19]:
import os
import glob
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import xgboost as xgb
import librosa
from scipy.signal import find_peaks
from scipy.ndimage import median_filter
import subprocess


# =============================================================================
# CONFIGURATION
# =============================================================================
BASE_DIR = r"E:\Research Project (Prof. Preeti Rao)\Top 100 files by Wheeze count"
SMILE_PATH = r"E:\Research Project (Prof. Preeti Rao)\opensmile-3.0-win-x64\bin\SMILExtract.exe"
PITCH_CONFIG = r"E:\Research Project (Prof. Preeti Rao)\opensmile-3.0-win-x64\myconfig\pitch.conf"
OPENSMILE_OUT = r"E:\Research Project (Prof. Preeti Rao)\Progress_2026\features_pitch_new"

os.makedirs(OPENSMILE_OUT, exist_ok=True)

WAV_DIR = os.path.join(BASE_DIR, "*.wav")
TXT_DIR = os.path.join(BASE_DIR, "*_label_audacity.txt")

OUT_TRAIN = "csv_train_combined_mel_15_1"
OUT_TEST = "csv_test_combined_mel_15_1"
os.makedirs(OUT_TRAIN, exist_ok=True)
os.makedirs(OUT_TEST, exist_ok=True)

SMOOTH_WIN = 7

SR = 16000
WIN_DUR = 0.050
HOP_DUR = 0.010
FREQ_MAX = 2000
N_MELS = 40


# =============================================================================
# FILE PAIRING
# =============================================================================
wav_files = sorted(glob.glob(WAV_DIR))
txt_files = sorted(glob.glob(TXT_DIR))

wav_stems = {Path(w).stem: w for w in wav_files}
txt_stems = {Path(t).stem.replace('_label_audacity', ''): t for t in txt_files}

pairs = [(wav_stems[k], txt_stems[k]) for k in wav_stems if k in txt_stems]
train_pairs, test_pairs = train_test_split(pairs, test_size=0.2, random_state=42)


# =============================================================================
# LABEL UTILITIES
# =============================================================================
def parse_audacity_labels(txt_path):
    labels = []
    with open(txt_path) as f:
        for line in f:
            p = line.strip().split('\t')
            if len(p) == 3:
                try:
                    s, e = float(p[0]), float(p[1])
                    lab = 1 if 'Wheeze' in p[2] else 0
                    labels.append((s, e, lab))
                except ValueError:
                    pass
    return labels


def wheeze_overlap_ratio(labels, t_start, t_end, min_ratio=0.4):
    overlap = 0.0
    dur = t_end - t_start
    for s, e, lab in labels:
        if lab == 1:
            overlap += max(0, min(e, t_end) - max(s, t_start))
    return 1 if (overlap / dur) >= min_ratio else 0


def enforce_min_wheeze_duration(labels, min_len=10):
    labels = labels.copy()
    n = len(labels)
    i = 0
    while i < n:
        if labels[i] == 1:
            start = i
            while i < n and labels[i] == 1:
                i += 1
            end = i
            if (end - start) < min_len:
                labels[start:end] = 0
        else:
            i += 1
    return labels


def fill_short_zero_gaps(labels, max_gap=4):
    labels = labels.copy()
    n = len(labels)
    i = 0
    while i < n:
        if labels[i] == 0:
            start = i
            while i < n and labels[i] == 0:
                i += 1
            end = i
            if start > 0 and end < n:
                if labels[start - 1] == 1 and labels[end] == 1:
                    if (end - start) <= max_gap:
                        labels[start:end] = 1
        else:
            i += 1
    return labels

def suppress_segments_without_low_peaks(preds, n_peaks, amp,
                                        peak_thresh=4, amp_thresh=8):
    """
    Suppress wheeze segments if NO frame inside the segment satisfies:
        n_peaks < peak_thresh AND amp < amp_thresh
    """
    preds = preds.copy()
    n = len(preds)
    i = 0

    while i < n:
        if preds[i] == 1:
            start = i
            while i < n and preds[i] == 1:
                i += 1
            end = i

            # Frame-wise condition
            condition = np.logical_and(
                n_peaks[start:end] < peak_thresh,
                amp[start:end] > amp_thresh
            )

            # If NO frame satisfies condition → suppress
            if not np.any(condition):
                preds[start:end] = 0
        else:
            i += 1

    return preds



# =============================================================================
# openSMILE pitch extraction
# =============================================================================
PITCH_FEATURES = [
    "F0Cand[0]", "F0Cand[1]",
    "candVoicing[0]", "candVoicing[1]",
    "candScores[0]", "candScores[1]",
    "nCandidates",
    "candScores[2]", "candVoicing[2]", "F0Cand[2]"
]


def extract_pitch_opensmile(wav_path):
    out_csv = os.path.join(OPENSMILE_OUT, Path(wav_path).stem + ".csv")

    if not os.path.exists(out_csv):
        cmd = [SMILE_PATH, "-C", PITCH_CONFIG, "-I", wav_path, "-O", out_csv]
        subprocess.run(cmd, capture_output=True)

        df = pd.read_csv(out_csv, sep=';')
        df.to_csv(out_csv, index=False)

    return pd.read_csv(out_csv)


# =============================================================================
# FEATURE EXTRACTION (FFT + MEL + PITCH)
# =============================================================================
def extract_per_file(wav_path, txt_path):
    audio, _ = librosa.load(wav_path, sr=SR)
    labels = parse_audacity_labels(txt_path)

    pitch_df = extract_pitch_opensmile(wav_path)
    time_col = [c for c in pitch_df.columns if "time" in c.lower()][0]
    pitch_df = pitch_df.rename(columns={time_col: "time_s"})

    win_len = int(WIN_DUR * SR)
    hop_len = int(HOP_DUR * SR)

    # ---- Mel spectrogram (log) ----
    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=SR,
        n_fft=win_len,
        hop_length=hop_len,
        n_mels=N_MELS,
        fmax=FREQ_MAX
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)

    rows = []

    for i, start in enumerate(range(0, len(audio) - win_len + 1, hop_len)):
        end = start + win_len
        frame = audio[start:end] * np.hanning(win_len)

        X = np.fft.rfft(frame)
        freqs = np.fft.rfftfreq(win_len, 1 / SR)
        mag = np.abs(X)
        mag[0] = 0

        valid = freqs <= FREQ_MAX
        freqs, mag = freqs[valid], mag[valid]

        total_energy = np.sum(mag ** 2) + 1e-12

        # ---- Normalized peak amplitude ----
        idx = np.argmax(mag)
        amp = mag[idx] / total_energy

        # ---- Wheeze band energy (100 Hz – 1 kHz) ----
        wheeze_band = (freqs >= 100) & (freqs <= 1000)
        band_energy = np.sum((mag[wheeze_band]) ** 2)

        band_energy_ratio = band_energy / total_energy

        p = mag / (np.sum(mag) + 1e-12)
        spec_entropy = -np.sum(p * np.log(p + 1e-12))

        centroid = np.sum(freqs * mag) / (np.sum(mag) + 1e-12)
        spec_bandwidth = np.sqrt(
            np.sum(((freqs - centroid) ** 2) * mag) / (np.sum(mag) + 1e-12)
        )

        peak_to_mean = amp / (np.mean(mag) + 1e-12)
        peaks, _ = find_peaks(mag, height=0.3 * amp)
        n_peaks = len(peaks)

        # ---- Mel features (frame-aligned) ----
        if i < mel_db.shape[1]:
            mel_frame = mel_db[:, i]
            mel_mean = np.mean(mel_frame)
            mel_std = np.std(mel_frame)
            mel_max = np.max(mel_frame)

            p_mel = np.exp(mel_frame)
            p_mel /= (np.sum(p_mel) + 1e-12)
            mel_entropy = -np.sum(p_mel * np.log(p_mel + 1e-12))
        else:
            mel_mean = mel_std = mel_max = mel_entropy = 0.0

        t_start = start / SR
        t_end = end / SR
        label = wheeze_overlap_ratio(labels, t_start, t_end)

        pitch_row = pitch_df.iloc[min(i, len(pitch_df) - 1)]
        pitch_values = [pitch_row[f] if f in pitch_df.columns else 0.0 for f in PITCH_FEATURES]

        rows.append([
            Path(wav_path).name, t_start,
            amp, band_energy_ratio,
            spec_entropy, spec_bandwidth,
            peak_to_mean, n_peaks,
            mel_mean, mel_std, mel_max, mel_entropy,
            *pitch_values,
            label
        ])

    return pd.DataFrame(rows, columns=[
        "file", "time_step_s",
        "amplitude", "band_energy_ratio",
        "spec_entropy", "spec_bandwidth",
        "peak_to_mean", "n_peaks",
        "mel_mean", "mel_std", "mel_max", "mel_entropy",
        *PITCH_FEATURES,
        "label"
    ])


# =============================================================================
# TRAIN / TEST PROCESSING
# =============================================================================
train_frames = []
for wav, txt in train_pairs:
    df = extract_per_file(wav, txt)
    df.to_csv(os.path.join(OUT_TRAIN, Path(wav).stem + "_train.csv"), index=False)
    train_frames.append(df)

train_df = pd.concat(train_frames, ignore_index=True)

test_frames = []
for wav, txt in test_pairs:
    df = extract_per_file(wav, txt)
    test_frames.append(df)

test_df = pd.concat(test_frames, ignore_index=True)


# =============================================================================
# MODEL TRAINING
# =============================================================================
feature_cols = [
    "amplitude", "band_energy_ratio",
    "spec_entropy", "spec_bandwidth",
    "peak_to_mean", "n_peaks",
    "mel_mean", "mel_std", "mel_max", "mel_entropy",
    *PITCH_FEATURES
]

X_train = train_df[feature_cols].values
y_train = train_df["label"].values
X_test = test_df[feature_cols].values
y_test = test_df["label"].values

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.08,
    subsample=0.85,
    colsample_bytree=0.85,
    eval_metric="logloss",
    random_state=42
)

model.fit(X_train_s, y_train)


# =============================================================================
# PREDICTION + SMOOTHING
# =============================================================================
test_df["wheeze_prob"] = model.predict_proba(X_test_s)[:, 1]

# test_df["wheeze_prob_smooth"] = (
#     test_df.groupby("file")["wheeze_prob"]
#     .transform(lambda x: median_filter(x, size=SMOOTH_WIN))
# )

test_df["predicted_label_raw"] = (
    test_df["wheeze_prob"] > 0.3
).astype(int)

# ---- Apply temporal constraints per file ----
final_preds = []

for fname, df_f in test_df.groupby("file"):
    preds = df_f["predicted_label_raw"].values
    peaks = df_f["n_peaks"].values
    amp= df_f["amplitude"].values

    # preds = enforce_min_wheeze_duration(preds, min_len=4)
    preds = enforce_min_wheeze_duration(preds, min_len=4)

    # Fill short zero gaps (<= 40 ms)
    preds = fill_short_zero_gaps(preds, max_gap=4)
    
    # Minimum wheeze duration (>= 100 ms)
    preds = enforce_min_wheeze_duration(preds, min_len=15)

    # NEW BIAS: suppress segments without low-peak frames
    # preds = suppress_segments_without_low_peaks(preds,peaks,amp,peak_thresh=4,amp_thresh=5)


    # Re-enforce duration (robustness)
    # preds = enforce_min_wheeze_duration(preds, min_len=10)

    final_preds.append(pd.Series(preds, index=df_f.index))

test_df["predicted_label"] = pd.concat(final_preds).sort_index()

for fname, df_f in test_df.groupby("file"):
    out = os.path.join(OUT_TEST, Path(fname).stem + "_test.csv")
    df_f.to_csv(out, index=False)

# =============================================================================
# METRICS
# =============================================================================
print(classification_report(y_test, test_df["predicted_label"],
                            target_names=["Normal", "Wheeze"]))
print("Confusion Matrix:\n",
      confusion_matrix(y_test, test_df["predicted_label"]))

print("\nDONE")

              precision    recall  f1-score   support

      Normal       0.75      0.59      0.66     10819
      Wheeze       0.79      0.89      0.84     19101

    accuracy                           0.78     29920
   macro avg       0.77      0.74      0.75     29920
weighted avg       0.78      0.78      0.78     29920

Confusion Matrix:
 [[ 6432  4387]
 [ 2112 16989]]

DONE


In [4]:
import torch
from torch.utils.data import Dataset
import numpy as np

class WheezeSequenceDataset(Dataset):
    def __init__(self, df, feature_cols, seq_len=50):
        self.seq_len = seq_len
        self.feature_cols = feature_cols
        self.samples = []

        for _, g in df.groupby("file"):
            X = g[feature_cols].values
            y = g["label"].values

            for i in range(len(g) - seq_len):
                x_seq = X[i:i+seq_len]
                y_seq = y[i + seq_len // 2]  # center frame
                self.samples.append((x_seq, y_seq))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x, y = self.samples[idx]
        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32)
        )


In [5]:
import torch.nn as nn

class WheezeCNN(nn.Module):
    def __init__(self, n_features):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv1d(n_features, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),

            nn.Linear(128, 1)
        )

    def forward(self, x):
        # x: (batch, time, features)
        x = x.transpose(1, 2)  # → (batch, features, time)
        return self.net(x).squeeze(1)


In [6]:
from torch.utils.data import DataLoader

SEQ_LEN = 50
BATCH_SIZE = 256

feature_cols = [
    "amplitude", "band_energy_ratio",
    "spec_entropy", "spec_bandwidth",
    "peak_to_mean", "n_peaks",
    "mel_mean", "mel_std", "mel_max", "mel_entropy",
    *PITCH_FEATURES
]

train_dataset = WheezeSequenceDataset(
    train_df, feature_cols, seq_len=SEQ_LEN
)

test_dataset = WheezeSequenceDataset(
    test_df, feature_cols, seq_len=SEQ_LEN
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True
)

test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False
)


In [7]:
import torch

y_train_seq = np.array([y for _, y in train_dataset.samples])

n_pos = np.sum(y_train_seq == 1)
n_neg = np.sum(y_train_seq == 0)

pos_weight = torch.tensor(n_neg / n_pos)
print("pos_weight:", pos_weight.item())

pos_weight: 0.8309012060396949


In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = WheezeCNN(n_features=len(feature_cols)).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 20

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for X, y in train_loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss/len(train_loader):.4f}")


Epoch 1/20, Loss: 0.4743
Epoch 2/20, Loss: 0.4169
Epoch 3/20, Loss: 0.3977
Epoch 4/20, Loss: 0.3818
Epoch 5/20, Loss: 0.3662
Epoch 6/20, Loss: 0.3555
Epoch 7/20, Loss: 0.3434
Epoch 8/20, Loss: 0.3330
Epoch 9/20, Loss: 0.3221
Epoch 10/20, Loss: 0.3106
Epoch 11/20, Loss: 0.3029
Epoch 12/20, Loss: 0.2946
Epoch 13/20, Loss: 0.2866
Epoch 14/20, Loss: 0.2778
Epoch 15/20, Loss: 0.2722
Epoch 16/20, Loss: 0.2664
Epoch 17/20, Loss: 0.2580
Epoch 18/20, Loss: 0.2524
Epoch 19/20, Loss: 0.2463
Epoch 20/20, Loss: 0.2428


In [13]:
model.eval()
all_probs = []
all_labels = []

with torch.no_grad():
    for X, y in test_loader:
        X = X.to(device)
        logits = model(X)
        probs = torch.sigmoid(logits)

        all_probs.append(probs.cpu().numpy())
        all_labels.append(y.numpy())

y_prob = np.concatenate(all_probs)
y_true = np.concatenate(all_labels)

THRESH = 0.3  # tune 0.5–0.7
y_pred = (y_prob > THRESH).astype(int)

In [14]:
from sklearn.metrics import classification_report, confusion_matrix


print(classification_report(
    y_true, y_pred, target_names=["Normal", "Wheeze"]
))

print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

              precision    recall  f1-score   support

      Normal       0.56      0.55      0.56     10164
      Wheeze       0.76      0.76      0.76     18756

    accuracy                           0.69     28920
   macro avg       0.66      0.66      0.66     28920
weighted avg       0.69      0.69      0.69     28920

Confusion Matrix:
 [[ 5633  4531]
 [ 4432 14324]]


In [15]:
import numpy as np
import torch
from sklearn.metrics import classification_report, confusion_matrix

# =============================================================================
# TEMPORAL EXTENT BIAS FUNCTIONS
# =============================================================================
def enforce_min_wheeze_duration(labels, min_len=10):
    labels = labels.copy()
    n = len(labels)
    i = 0
    while i < n:
        if labels[i] == 1:
            start = i
            while i < n and labels[i] == 1:
                i += 1
            end = i
            if (end - start) < min_len:
                labels[start:end] = 0
        else:
            i += 1
    return labels


def fill_short_zero_gaps(labels, max_gap=4):
    labels = labels.copy()
    n = len(labels)
    i = 0
    while i < n:
        if labels[i] == 0:
            start = i
            while i < n and labels[i] == 0:
                i += 1
            end = i
            if start > 0 and end < n:
                if labels[start - 1] == 1 and labels[end] == 1:
                    if (end - start) <= max_gap:
                        labels[start:end] = 1
        else:
            i += 1
    return labels


# =============================================================================
# MODEL INFERENCE
# =============================================================================
model.eval()

all_probs = []
all_labels = []

with torch.no_grad():
    for X, y in test_loader:
        X = X.to(device)
        logits = model(X)
        probs = torch.sigmoid(logits).view(-1)

        all_probs.append(probs.cpu().numpy())
        all_labels.append(y.numpy())

y_prob = np.concatenate(all_probs)
y_true = np.concatenate(all_labels)

# =============================================================================
# THRESHOLDING
# =============================================================================
THRESH = 0.1
y_pred_raw = (y_prob > THRESH).astype(int)

# =============================================================================
# TEMPORAL EXTENT BIAS (GLOBAL SEQUENCE)
# =============================================================================
# Assumes sequential frames (no shuffle)
y_pred = enforce_min_wheeze_duration(y_pred_raw, min_len=3)
y_pred = fill_short_zero_gaps(y_pred, max_gap=3)
y_pred = enforce_min_wheeze_duration(y_pred, min_len=15)

# =============================================================================
# METRICS
# =============================================================================
print(classification_report(
    y_true,
    y_pred,
    target_names=["Normal", "Wheeze"]
))

print("Confusion Matrix:\n",
      confusion_matrix(y_true, y_pred))


              precision    recall  f1-score   support

      Normal       0.57      0.31      0.40     10164
      Wheeze       0.70      0.88      0.78     18756

    accuracy                           0.68     28920
   macro avg       0.64      0.59      0.59     28920
weighted avg       0.65      0.68      0.64     28920

Confusion Matrix:
 [[ 3102  7062]
 [ 2322 16434]]


In [16]:
model.eval()

train_probs = []
train_labels = []

with torch.no_grad():
    for X, y in train_loader:
        X = X.to(device)
        logits = model(X)
        probs = torch.sigmoid(logits)

        train_probs.append(probs.cpu().numpy())
        train_labels.append(y.numpy())

y_train_prob = np.concatenate(train_probs)
y_train_true = np.concatenate(train_labels)

THRESH = 0.2  # same as test
y_train_pred = (y_train_prob > THRESH).astype(int)

print("CNN TRAIN METRICS")
print(classification_report(
    y_train_true,
    y_train_pred,
    target_names=["Normal", "Wheeze"]
))

print("Confusion Matrix:\n",
      confusion_matrix(y_train_true, y_train_pred))


CNN TRAIN METRICS
              precision    recall  f1-score   support

      Normal       0.88      0.59      0.71     52498
      Wheeze       0.73      0.93      0.82     63182

    accuracy                           0.78    115680
   macro avg       0.81      0.76      0.77    115680
weighted avg       0.80      0.78      0.77    115680

Confusion Matrix:
 [[31057 21441]
 [ 4124 59058]]
